# MediBot — Component 3: Reranking with a Cross-Encoder

Hybrid retrieval (Component 2) fuses two **independent** scoring functions — dense cosine similarity and BM25 term overlap. Neither ever looks at the query and a candidate chunk *together*; they each score in isolation and get combined by rank position (RRF). A cross-encoder is different: it takes `(query, chunk_text)` as one joint input and outputs a single relevance score, which is a strictly stronger (and slower — it can't be precomputed) signal.

1. Run hybrid retrieval for a broad candidate set (top-10, from Component 2)
2. Rerank all 10 with a cross-encoder, narrowing to top-3
3. Confirm the reranker actually *reorders* results (not just relabels them)
4. Confirm only the reranked top-3 — not the full top-10 — reach the LLM
5. Generate the final answer from the reranked chunks only

## 1 — Imports + connect to the collection

In [1]:
import logging
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "medibot"))

from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

from config import (
    COLLECTION_NAME,
    EMBED_MODEL,
    HYBRID_TOP_K,
    QDRANT_PATH,
    RERANK_MODEL,
    RERANK_TOP_K,
    SPARSE_EMBED_MODEL,
)
from retrieval import generate_answer, hybrid_search
from rerank import get_cross_encoder, rerank

# INFO so rerank.py's per-candidate score logging (the whole point of this notebook) is visible.
logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("medibot.rerank").setLevel(logging.INFO)

client = QdrantClient(path=QDRANT_PATH)
print(f"Connected. Points in '{COLLECTION_NAME}': {client.count(collection_name=COLLECTION_NAME).count}")

/Users/vennilave23/Documents/VSCode_Workspace/CodeBasics/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RAG assignment dir: /Users/vennilave23/Documents/VSCode_Workspace/CodeBasics/Assignments/2_RAG
Data dir: /Users/vennilave23/Documents/VSCode_Workspace/CodeBasics/Assignments/2_RAG/Medibot_Assignment_Resources/mediassist_data
Connected. Points in 'medibot_docs': 252


## 2 — Load the embedders + cross-encoder

In [2]:
dense_embedder = SentenceTransformer(EMBED_MODEL)
sparse_embedder = SparseTextEmbedding(model_name=SPARSE_EMBED_MODEL)
cross_encoder = get_cross_encoder()

print(f"Dense embedder:  {EMBED_MODEL}")
print(f"Sparse embedder: {SPARSE_EMBED_MODEL}")
print(f"Cross-encoder:   {RERANK_MODEL}")
print(f"Hybrid fetches top-{HYBRID_TOP_K} candidates; reranker narrows to top-{RERANK_TOP_K}")

No device provided, using mps


HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"


...

HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-6-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"


HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-6-v2/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"


HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


Dense embedder:  sentence-transformers/all-MiniLM-L6-v2
Sparse embedder: Qdrant/bm25
Cross-encoder:   cross-encoder/ms-marco-MiniLM-L-6-v2
Hybrid fetches top-10 candidates; reranker narrows to top-3


## 3 — Step 1: broad hybrid candidate set (top-10)

In [3]:
query = "What is the pre-authorization process for a claim?"
role = "billing_executive"

hybrid_results = hybrid_search(client, COLLECTION_NAME, dense_embedder, sparse_embedder, query, role, limit=HYBRID_TOP_K)
candidates = [p.payload for p in hybrid_results.points]

print(f"Query: {query!r}  (role={role})\n")
print(f"Hybrid (fused RRF) order -- {len(candidates)} candidates:\n")
for i, c in enumerate(candidates, 1):
    print(f"  {i:2d}. {c['section_title']}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.70it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.70it/s]

Query: 'What is the pre-authorization process for a claim?'  (role=billing_executive)

Hybrid (fused RRF) order -- 10 candidates:

   1. 3.3 Process
   2. 1.1 Pre-authorisation timeline
   3. 3.1 When to raise
   4. 1.3 Step-by-step
   5. 1.4 Typical approval turnaround (initial pre-auth)
   6. See also
   7. Purpose & Scope
   8. 3. Pre-Authorisation Enhancement
   9. 1.2 Documents required for pre-authorisation
  10. 3.2 Documents required


## 4 — Step 2: rerank all 10 jointly, narrow to top-3

`rerank()` scores *every* candidate (not just the ones it keeps) and logs each score — this is the assignment's tip: log reranker scores during development so you can see the reordering happen, not just trust that it did.

In [4]:
reranked = rerank(query, candidates, top_k=RERANK_TOP_K)

print(f"\nKept top-{RERANK_TOP_K} after reranking:\n")
for i, c in enumerate(reranked, 1):
    print(f"  {i}. score={c['rerank_score']:.3f}  {c['section_title']}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  9.35it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  9.32it/s]


rerank #1 score=3.5496  [billing] claim_submission_guide.md > 1.1 Pre-authorisation timeline


rerank #2 score=2.3409  [billing] claim_submission_guide.md > 3.3 Process


...

rerank #7 score=0.1978  [billing] claim_submission_guide.md > 3. Pre-Authorisation Enhancement


rerank #8 score=-0.3047  [billing] claim_submission_guide.md > 1.3 Step-by-step


rerank #9 score=-1.4663  [billing] claim_submission_guide.md > 3.2 Documents required


rerank #10 score=-1.9398  [billing] claim_submission_guide.md > Purpose & Scope



Kept top-3 after reranking:

  1. score=3.550  1.1 Pre-authorisation timeline
  2. score=2.341  3.3 Process
  3. score=1.892  1.4 Typical approval turnaround (initial pre-auth)


## 5 — Confirm the reranker actually reordered results

If the cross-encoder just agreed with the hybrid fusion order, reranking would be pointless. Compare each chunk's hybrid rank (position in the top-10) against its reranked position.

In [5]:
hybrid_rank_by_section = {c["section_title"]: i for i, c in enumerate(candidates, 1)}

print(f"{'section':<45} {'hybrid rank':>11} {'rerank rank':>11}")
for new_rank, c in enumerate(reranked, 1):
    old_rank = hybrid_rank_by_section[c["section_title"]]
    flag = "  <-- moved up" if old_rank > new_rank else ("" if old_rank == new_rank else "  <-- moved down")
    print(f"{c['section_title']:<45} {old_rank:>11} {new_rank:>11}{flag}")

top1_changed = reranked[0]["section_title"] != candidates[0]["section_title"]
print(f"\nHybrid's #1 result was: {candidates[0]['section_title']!r}")
print(f"Reranked #1 result is:  {reranked[0]['section_title']!r}")
print(f"Reranking changed the top result: {top1_changed}")

section                                       hybrid rank rerank rank
1.1 Pre-authorisation timeline                          2           1  <-- moved up
3.3 Process                                             1           2  <-- moved down
1.4 Typical approval turnaround (initial pre-auth)           5           3  <-- moved up

Hybrid's #1 result was: '3.3 Process'
Reranked #1 result is:  '1.1 Pre-authorisation timeline'
Reranking changed the top result: True


## 6 — Confirm the full candidate set never reaches the LLM

Per the assignment: only the reranked top-k may be included in the prompt. We build both prompts' context and check the discarded hybrid candidates are absent from what actually goes to `generate_answer()`.

In [6]:
kept_titles = {c["section_title"] for c in reranked}
discarded = [c for c in candidates if c["section_title"] not in kept_titles]

print(f"Hybrid candidates fetched: {len(candidates)}")
print(f"Chunks passed to the LLM after reranking: {len(reranked)}")
print(f"Discarded (never reach the prompt): {len(discarded)}")
for c in discarded:
    print(f"  - {c['section_title']}")

assert len(reranked) == RERANK_TOP_K, "Reranker did not narrow to the configured top_k"
assert len(discarded) == len(candidates) - RERANK_TOP_K

Hybrid candidates fetched: 10
Chunks passed to the LLM after reranking: 3
Discarded (never reach the prompt): 7
  - 3.1 When to raise
  - 1.3 Step-by-step
  - See also
  - Purpose & Scope
  - 3. Pre-Authorisation Enhancement
  - 1.2 Documents required for pre-authorisation
  - 3.2 Documents required


## 7 — Generate the final answer from the reranked chunks only

In [7]:
answer = generate_answer(query, reranked)

print(f"Question ({role}): {query}\n")
print(f"Answer:\n{answer}\n")
print("Sources (reranked top-3 only):")
for c in reranked:
    print(f"  - {c['source_document']} > {c['section_title']} ({c['collection']}), rerank_score={c['rerank_score']:.3f}")

HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Question (billing_executive): What is the pre-authorization process for a claim?

Answer:
The pre‑authorisation process for a claim involves the following steps:

1. **Submit a pre‑auth request**  
   * For a planned admission, the request must be made **at least 48 hours before admission**.  
   * For an emergency admission, the request must be made **within 6 hours of admission** (missing this window is the most common reason for avoidable cashless denials)【claim_submission_guide.md > 1.1 Pre-authorisation timeline】.

2. **Enhance the pre‑auth if needed**  
   * Raise any enhancement **before the original approved amount is exhausted**—never after discharge【claim_submission_guide.md > 3.3 Process】.  
   * Submit the enhancement via the insurer portal linked to the original pre‑auth, marking it clearly as “Enhancement”【claim_submission_guide.md > 3.3 Process】.  
   * Track it as a child claim under the parent pre‑auth in MBP【claim_submission_guide.md > 3.3 Process】.  
   * If the enha

## 8 — Next: Component 4 (SQL RAG)

This notebook's `hybrid_search -> rerank -> generate_answer` chain is the document-RAG branch of the `/chat` routing logic. Component 4 adds the other branch: `sql_rag_chain(question) -> str` for analytical questions over `mediassist.db`, gated to `billing_executive`/`admin`.